## Load Necessary Tools
Vizualization References
[Pandas](http://pandas.pydata.org/pandas-docs/stable/visualization.html),
[Seaborn](https://seaborn.pydata.org/)

In [2]:
# Redshift Engine
from sqlalchemy import create_engine
from pandas import read_sql_query as qry
# Data Sci 
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import train_test_split
import tensorflow as tf
# Viz
%matplotlib inline
from matplotlib import pyplot as plt
from pandas.tools.plotting import scatter_matrix

## Connect to Redshift

In [3]:
db='<database>'
host='<redshift-host>'
user='<username>'
psswd='<password>'
conStr='postgresql://'+user+':'+psswd+'@'+host+':5439/'+db
con = create_engine(conStr)

## Import Data

In [4]:
# Big4 Dates
rumble = ['2015-01-25','2016-01-24','2017-01-29']
slam = ['2015-08-23','2016-08-21','2017-08-20']
survivor = ['2015-11-22','2016-11-20','2017-11-19']
mania = ['2015-03-29','2016-04-03','2017-04-02']

# Query!
df = qry("SELECT \
              as_on_dt, \
              cust_guid, \
              CASE \
                WHEN ram_country = 'us' THEN 1 \
                ELSE -1 \
                END AS domestic, \
              num_vol_losses, \
              pmt_pct_days_paid, \
              pct_ppv1, \
              num_invol_fail, \
              CASE \
                WHEN no_view_90days_flag = 0 THEN -1 \
                ELSE no_view_90days_flag \
                END AS no_view_90days_flag, \
              vod_lst_30days, \
              nxt_view_ep_lst_90days, \
              CASE \
                WHEN (as_on_dt BETWEEN '"+rumble[0]+"'::date-'1 month'::interval \
                    AND '"+rumble[0]+"'::date) \
                    OR (as_on_dt BETWEEN '"+rumble[1]+"'::date-'1 month'::interval \
                    AND '"+rumble[1]+"'::date) \
                    OR (as_on_dt BETWEEN '"+rumble[2]+"'::date-'1 month'::interval \
                    AND '"+rumble[2]+"'::date) \
                  THEN 1 \
                  ELSE -1 \
                  END AS rumble30, \
              CASE  \
                WHEN (as_on_dt BETWEEN '"+slam[0]+"'::date-'1 month'::interval \
                    AND '"+slam[0]+"'::date) \
                    OR (as_on_dt BETWEEN '"+slam[1]+"'::date-'1 month'::interval \
                    AND '"+slam[1]+"'::date) \
                    OR (as_on_dt BETWEEN '"+slam[2]+"'::date-'1 month'::interval \
                    AND '"+slam[2]+"'::date) \
                  THEN 1 \
                  ELSE -1 \
                  END AS slam30, \
              CASE  \
                WHEN (as_on_dt BETWEEN '"+survivor[0]+"'::date-'1 month'::interval \
                    AND '"+survivor[0]+"'::date) \
                    OR (as_on_dt BETWEEN '"+survivor[1]+"'::date-'1 month'::interval \
                    AND '"+survivor[1]+"'::date) \
                    OR (as_on_dt BETWEEN '"+survivor[2]+"'::date-'1 month'::interval \
                    AND '"+survivor[2]+"'::date) \
                  THEN 1 \
                  ELSE -1 \
                  END AS survivor30, \
              CASE  \
                 WHEN (as_on_dt BETWEEN '"+mania[0]+"'::date-'1 month'::interval \
                    AND '"+mania[0]+"'::date) \
                    OR (as_on_dt BETWEEN '"+mania[1]+"'::date-'1 month'::interval \
                    AND '"+mania[1]+"'::date) \
                    OR (as_on_dt BETWEEN '"+mania[2]+"'::date-'1 month'::interval \
                    AND '"+mania[2]+"'::date) \
                  THEN 1 \
                  ELSE -1 \
                  END AS mania30, \
              CASE \
                WHEN pmt1_aftr_as_on  = 0 THEN 1 \
                WHEN pmt1_aftr_as_on >= 1 THEN 0 \
                ELSE NULL \
                END AS churn \
          FROM <schema>.<churn_table> \
            WHERE current_state='Active Paid' \
              AND latest_active_period > 29 \
              AND current_rc_payer = 0 \
              AND pmt1_aftr_as_on IS NOT NULL \
         ",con)

## Churn Model Object

In [ ]:
class churn_data:
    
    def __init__(self, dfx, dfy, extra=None, split1=0.5, split2=0.5):
        _x = np.array(dfx, dtype=np.float32)
        _y = np.array([dfy], dtype=np.float32).T
        self.features = dfx.columns.tolist()
        self.event    = dfy.columns.tolist()
        self.x_train, self.y_train, \
            self.x_valid, self.y_valid, \
            self.x_test,  self.y_test = self.__split_data(_x,_y)
        self.n_data     = _x.shape[0]
        self.n_features = _x.shape[1]
        self.n_classes  = _y.shape[1]
        self.extra_info = extra
    
    
    #=========================================================
    def fill_nan():
        # Thus far, all inputs w/ NaNs have 0-fill NaN types, 
        #   so we can zero-fill everything... However, this might
        #   not always be the case.
        #
        df.fillna(0)
    
        return [x_data, y_data]

    
    #=========================================================
    def __split_data(x_data, y_data, split1=0.5, split2=0.5):
        # Default Training-Validation-Test: 50%, 25%, 25%
        #   -- Training          = 1.0 - split1
        #   -- Validation + Test = split1
        #   -- Validation        = split1*(1.0 - split2)
        #   -- Test              = split1*split2
        x_train, x_vt, y_train, y_vt = train_test_split(x_data, y_data, 
                                            test_size = split1, random_state=42)
        x_valid, x_test, y_valid, y_test = train_test_split(x_vt, y_vt, 
                                            test_size = split2, random_state=31)
        return(x_train, y_train, x_valid, y_valid, x_test, y_test)

    
    #=========================================================
    def raw_churn_data(df, x_cols, y_cols):
    
        # Splits
        x_splits, y_splits = split_data(x_data, y_data)
    
        # Output
        return(
            {
                'x_train': x_splits[0], 'x_valid': x_splits[1], 'x_test': x_splits[2],
                'y_train': y_splits[0], 'y_valid': y_splits[1], 'y_test': y_splits[2]
            }
        )
    
    
    #========================================================
    def zscore(xtrain, xval, xtest):
    # vec can be np.array or pd.DataFrame
    mmm = xtrain.mean()
    sss = xtrain.std()
    return {'loc': mmm, 'scale': sss,
      'train':  (xtrain - mmm)/sss,
      'val':    (xval - mmm)/sss,
      'test':   (xtest - mmm)/sss}
    
    #========================================================
    def halfz(xtrain, xval, xtest):
    # vec can be np.array or pd.DataFrame
    mmm = xtrain.mean()
    sss = xtrain.std()
    return {'loc': mmm, 'scale': sss,
      'train': 0.5*(xtrain - mmm)/sss,
      'val': 0.5*(xval - mmm)/sss,
      'test': 0.5*(xtest - mmm)/sss}

    #========================================================
    def fn01(xtrain, xval, xtest):
        # vec should be 1D np.array or pd.DataFrame
        mn = xtrain.min()
        mx = xtrain.max()
        return {'mn':mn, 'mx':mx,
                'train': (xtrain - mn)/(mx-mn),
                'val':   (xval   - mn)/(mx-mn),
                'test':  (xtest  - mn)/(mx-mn)}

    #========================================================
    def fn11(xtrain, xval, xtest):
        # vec should be 1D np.array or pd.DataFrame
        mn = xtrain.min()
        mx = xtrain.max()
        return {'mn':mn, 'mx':mx,
                'train': (2*xtrain - (mx+mn))/(mx-mn),
                'val':   (2*xval   - (mx+mn))/(mx-mn),
                'test':  (2*xtest  - (mx+mn))/(mx-mn)}
